# Skill Catalog Synchronization

This notebook synchronizes skill taxonomies from external sources (O*NET, ESCO, proprietary catalogs) and maintains a canonical skill catalog with mappings, aliases, and metadata.

## Architecture

**Input**: Canonical skills CSV (`canonical_skills.csv`), optional external sources  
**Output**: `workspace.intermediate.inter_skill_catalog`  
**Evidence**: `workspace.intermediate.inter_job_skill_evidence`  
**Mode**: Incremental sync (adds new skills, preserves existing)

## Sync Processing

- Tracks sync runs in `metadata.skill_catalog_sync_runs`
- Idempotent: safe to re-run
- MERGE operation preserves existing skills and audit timestamps
- Each skill tracks created_at and updated_at

## Features
- Multi-source catalog integration (O*NET, ESCO, LinkedIn Skills)
- Automatic duplicate detection and merging
- Skill versioning and change tracking
- Audit trail for all catalog updates

In [0]:
dbutils.widgets.text("sync_version", "", "Sync Version (optional, leave empty for auto)")

sync_version = dbutils.widgets.get("sync_version").strip()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, ArrayType, TimestampType
from datetime import datetime
import json

CATALOG = "workspace"
INTERMEDIATE_SCHEMA = f"{CATALOG}.intermediate"
METADATA_SCHEMA = f"{CATALOG}.metadata"
SILVER_SCHEMA = f"{CATALOG}.silver"

# Configuration
CONFIG = {
    "skill_catalog_table": f"{INTERMEDIATE_SCHEMA}.inter_skill_catalog",
    "skill_evidence_table": f"{INTERMEDIATE_SCHEMA}.inter_job_skill_evidence",
    "silver_jobs_table": f"{SILVER_SCHEMA}.silver_jobs_current",
    "external_sources": {
        "onet": {
            "enabled": False,
            "path": "/mnt/external/onet/skills.json",
            "priority": 1
        },
        "esco": {
            "enabled": False,
            "path": "/mnt/external/esco/skills.csv",
            "priority": 2
        },
        "linkedin": {
            "enabled": False,
            "path": "/mnt/external/linkedin/skills.parquet",
            "priority": 3
        }
    },
    "similarity_threshold": 0.85,
    "auto_merge_threshold": 0.95
}

# Generate sync run ID
if not sync_version:
    sync_version = datetime.now().strftime("v%Y%m%d_%H%M%S")

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()
run_timestamp = F.current_timestamp()

print(f"Sync Version: {sync_version}")
print(f"Run ID: {run_id}")
print("\nSkill Catalog Sync Configuration:")
print(json.dumps(CONFIG, indent=2))

In [0]:
# Create metadata table to track skill catalog sync runs
metadata_table = f"{METADATA_SCHEMA}.skill_catalog_sync_runs"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  sync_version STRING,
  run_id STRING,
  skills_before INT,
  skills_after INT,
  new_skills_added INT,
  skills_with_aliases INT,
  external_sources_loaded INT,
  synced_at TIMESTAMP,
  status STRING
)
USING DELTA
COMMENT 'Tracks skill catalog sync runs'
""")

# Define metadata schema
metadata_schema = StructType([
    StructField("sync_version", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("skills_before", IntegerType(), True),
    StructField("skills_after", IntegerType(), True),
    StructField("new_skills_added", IntegerType(), True),
    StructField("skills_with_aliases", IntegerType(), True),
    StructField("external_sources_loaded", IntegerType(), True),
    StructField("synced_at", TimestampType(), True),
    StructField("status", StringType(), True)
])

print(f"Metadata table: {metadata_table}")

In [0]:
# ⚠️ UNCOMMENT TO RESET: Deletes all skills and metadata to rebuild from scratch
# Use this when the canonical skills CSV has major structural changes

# print("⚠️  RESETTING: Deleting all skills and sync metadata...")
# spark.sql(f"DELETE FROM {CONFIG['skill_catalog_table']}")
# spark.sql(f"DELETE FROM {metadata_table}")
# print("✓ Reset complete. Catalog will be rebuilt from canonical sources.")

In [0]:
# Load existing skill catalog or create new one
try:
    skill_catalog_df = spark.table(CONFIG["skill_catalog_table"])
    print(f"Loaded existing skill catalog with {skill_catalog_df.count()} skills")
    display(skill_catalog_df.limit(10))
    catalog_count = skill_catalog_df.count()
except Exception as e:
    print(f"Skill catalog table not found: {e}")
    print("Initializing new skill catalog...")
    
    # Create empty skill catalog with audit timestamps
    skill_catalog_df = spark.createDataFrame([], StructType([
        StructField("canonical_skill_id", StringType(), False),
        StructField("skill_name", StringType(), False),
        StructField("skill_category", StringType(), True),
        StructField("aliases", ArrayType(StringType()), True),
        StructField("active_flag", StringType(), True),
        StructField("taxonomy_version", StringType(), False),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ]))
    catalog_count = 0
    print("Empty catalog initialized")

In [0]:
# Load skills from external sources
external_skills = []

for source_name, source_config in CONFIG["external_sources"].items():
    if not source_config["enabled"]:
        print(f"Source '{source_name}' is disabled, skipping...")
        continue
    
    try:
        print(f"Loading skills from {source_name}...")
        source_path = source_config["path"]
        
        # Load based on file format
        if source_path.endswith(".json"):
            source_df = spark.read.json(source_path)
        elif source_path.endswith(".csv"):
            source_df = spark.read.csv(source_path, header=True)
        elif source_path.endswith(".parquet"):
            source_df = spark.read.parquet(source_path)
        else:
            print(f"  Unsupported format for {source_name}")
            continue
        
        # Standardize schema
        source_df = source_df.select(
            F.col("skill_name").alias("canonical_skill_name"),
            F.coalesce(F.col("category"), F.lit("Uncategorized")).alias("skill_category"),
            F.col("description"),
            F.lit(source_name).alias("source"),
            F.lit(source_config["priority"]).alias("priority")
        )
        
        count = source_df.count()
        print(f"  Loaded {count} skills from {source_name}")
        external_skills.append(source_df)
        
    except Exception as e:
        print(f"  Error loading {source_name}: {e}")

if external_skills:
    # Combine all external sources
    from functools import reduce
    all_external_df = reduce(lambda df1, df2: df1.union(df2), external_skills)
    print(f"\nTotal external skills loaded: {all_external_df.count()}")
    display(all_external_df.limit(10))
else:
    print("\nNo external sources enabled or loaded")
    all_external_df = None

In [0]:
# Load canonical skills from taxonomy table
print("Loading canonical skill taxonomy from metadata table...")

# Read directly from taxonomy_skill_catalog table
canonical_skills_df = spark.table(f"{METADATA_SCHEMA}.taxonomy_skill_catalog").select(
    F.col("canonical_skill").alias("skill_name"),
    F.col("skill_category"),
    F.coalesce(F.col("sector_key"), F.lit("CROSS_SECTOR")).alias("sector_key"),
    F.col("aliases")
).withColumn(
    "source", F.lit("canonical")
).withColumn(
    "priority", F.lit(100)  # Canonical skills have highest priority
)

print(f"Loaded {canonical_skills_df.count()} canonical skills from metadata")
print("\nSkills by category and sector:")
display(
    canonical_skills_df.groupBy("skill_category", "sector_key")
    .count()
    .orderBy("skill_category", F.desc("count"))
)

# In production, extracted skills from job postings would be matched against this canonical list
# For now, use canonical skills as the foundation
unique_extracted_df = canonical_skills_df

In [0]:
from pyspark.sql.functions import lower, trim, regexp_replace
import uuid

def normalize_skill_name(name):
    """Normalize skill name for comparison."""
    if not name:
        return ""
    return name.lower().strip().replace("  ", " ")

# Combine all skill sources
if all_external_df:
    # External + Extracted
    combined_df = all_external_df.select(
        F.col("canonical_skill_name").alias("skill_name"),
        "skill_category",
        "description",
        "source",
        "priority"
    ).union(
        unique_extracted_df.select(
            "skill_name",
            "skill_category",
            F.lit(None).alias("description"),
            "source",
            "priority"
        )
    )
else:
    # Only extracted
    combined_df = unique_extracted_df.select(
        "skill_name",
        "skill_category",
        F.lit(None).alias("description"),
        "source",
        "priority"
    )

print(f"Combined {combined_df.count()} skills from all sources")

# Normalize skill names for deduplication
combined_df = combined_df.withColumn(
    "normalized_name",
    lower(trim(regexp_replace(F.col("skill_name"), "\\s+", " ")))
)

# Deduplicate: keep skill with highest priority (lowest priority number)
from pyspark.sql.window import Window

window_spec = Window.partitionBy("normalized_name").orderBy(F.asc("priority"))

deduped_df = combined_df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
).filter(
    F.col("row_num") == 1
).drop("row_num")

dedup_count = deduped_df.count()
print(f"After deduplication: {dedup_count} unique skills")
print(f"Removed {combined_df.count() - dedup_count} duplicates")

display(deduped_df.limit(20))

In [0]:
# Generate skill IDs and add metadata
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def generate_skill_id(skill_name):
    """Generate deterministic skill ID based on normalized name."""
    normalized = normalize_skill_name(skill_name)
    return f"skill_{hash(normalized) % 10**10:010d}"

generate_skill_id_udf = udf(generate_skill_id, StringType())

# Prepare new skills for catalog (matching inter_skill_catalog schema)
new_skills_df = deduped_df.select(
    generate_skill_id_udf(F.col("skill_name")).alias("canonical_skill_id"),
    F.col("skill_name"),
    F.col("skill_category"),
    F.array().alias("aliases"),  # Empty array, will populate from aliases logic
    F.lit("TRUE").alias("active_flag"),
    F.lit(sync_version).alias("taxonomy_version"),
    run_timestamp.alias("created_at"),
    run_timestamp.alias("updated_at")
)

print(f"Prepared {new_skills_df.count()} skills for catalog")
display(new_skills_df.limit(10))

In [0]:
# Merge new skills with existing catalog
if catalog_count > 0:
    print("Merging with existing catalog...")
    
    # Find truly new skills (not in existing catalog)
    existing_skill_ids = skill_catalog_df.select("canonical_skill_id").distinct()
    
    truly_new_df = new_skills_df.join(
        existing_skill_ids,
        on="canonical_skill_id",
        how="left_anti"
    )
    
    # Add timestamps and reorder columns to match new_skills_df schema
    if "created_at" not in skill_catalog_df.columns:
        skill_catalog_df = skill_catalog_df.withColumn("created_at", run_timestamp)
    if "updated_at" not in skill_catalog_df.columns:
        skill_catalog_df = skill_catalog_df.withColumn("updated_at", run_timestamp)
    
    # Reorder columns to match new_skills_df for union compatibility
    skill_catalog_df = skill_catalog_df.select(
        "canonical_skill_id",
        "skill_name",
        "skill_category",
        "aliases",
        "active_flag",
        "taxonomy_version",
        "created_at",
        "updated_at"
    )
    
    new_count = truly_new_df.count()
    print(f"Found {new_count} new skills to add")
    
    if new_count > 0:
        # Combine existing + new
        updated_catalog_df = skill_catalog_df.union(truly_new_df)
        print(f"Updated catalog: {updated_catalog_df.count()} total skills")
    else:
        updated_catalog_df = skill_catalog_df
        print("No new skills to add")
else:
    print("No existing catalog, using new skills as catalog")
    updated_catalog_df = new_skills_df

print(f"\nFinal catalog size: {updated_catalog_df.count()} skills")
display(updated_catalog_df.limit(20))

In [0]:
# Generate aliases for skills (variations, abbreviations, etc.)
from pyspark.sql.functions import explode, array, lit

def generate_aliases(skill_name):
    """Generate common aliases for a skill."""
    aliases = []
    name = skill_name.strip()
    
    # Original name
    aliases.append((name, "original", 1.0))
    
    # Lowercase
    if name.lower() != name:
        aliases.append((name.lower(), "lowercase", 1.0))
    
    # Common abbreviations
    abbrev_map = {
        "JavaScript": "JS",
        "TypeScript": "TS",
        "Python": "Py",
        "Machine Learning": "ML",
        "Artificial Intelligence": "AI",
        "Natural Language Processing": "NLP",
        "Computer Vision": "CV",
        "Software Development": "SWE",
    }
    
    if name in abbrev_map:
        aliases.append((abbrev_map[name], "abbreviation", 0.9))
    
    return aliases

# Update skill catalog with aliases in the aliases array column
print("Generating skill aliases and updating catalog...")

# Collect aliases for each skill
from collections import defaultdict
alias_map = defaultdict(list)

for row in updated_catalog_df.limit(100).collect():
    skill_id = row["canonical_skill_id"]
    skill_name = row["skill_name"]
    
    for alias, alias_type, confidence in generate_aliases(skill_name):
        if alias != skill_name:  # Don't include the canonical name itself
            alias_map[skill_id].append(alias)

print(f"Generated aliases for {len(alias_map)} skills")

# Update catalog with aliases
if alias_map:
    alias_list = [(k, v) for k, v in alias_map.items()]
    aliases_df = spark.createDataFrame(alias_list, ["canonical_skill_id", "aliases"])
    
    # Join back to catalog and update aliases column
    updated_catalog_df = updated_catalog_df.drop("aliases").join(
        aliases_df,
        on="canonical_skill_id",
        how="left"
    ).withColumn(
        "aliases",
        F.coalesce(F.col("aliases"), F.array())
    )
    
    print(f"Updated {aliases_df.count()} skills with aliases")
    display(updated_catalog_df.filter(F.size(F.col("aliases")) > 0).limit(10))

In [0]:
# Write updated skill catalog using MERGE (idempotent)
print(f"Writing skill catalog to {CONFIG['skill_catalog_table']}...")

# Prepare for merge - update updated_at timestamp
merge_df = updated_catalog_df.withColumn(
    "updated_at",
    F.when(F.col("canonical_skill_id").isNotNull(), run_timestamp)
     .otherwise(F.col("updated_at"))
)

# Create temp view
merge_df.createOrReplaceTempView("skill_catalog_updates")

# Create table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONFIG['skill_catalog_table']} (
  canonical_skill_id STRING,
  skill_name STRING,
  skill_category STRING,
  aliases ARRAY<STRING>,
  active_flag STRING,
  taxonomy_version STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
""")

# Add timestamp columns if they don't exist (for existing tables from prior runs)
try:
    spark.sql(f"ALTER TABLE {CONFIG['skill_catalog_table']} ADD COLUMN created_at TIMESTAMP")
    print("Added created_at column to existing table")
except Exception:
    pass  # Column already exists

try:
    spark.sql(f"ALTER TABLE {CONFIG['skill_catalog_table']} ADD COLUMN updated_at TIMESTAMP")
    print("Added updated_at column to existing table")
except Exception:
    pass  # Column already exists

# MERGE: insert new skills, update existing
spark.sql(f"""
MERGE INTO {CONFIG['skill_catalog_table']} target
USING skill_catalog_updates source
ON target.canonical_skill_id = source.canonical_skill_id
WHEN MATCHED THEN UPDATE SET
  skill_name = source.skill_name,
  skill_category = source.skill_category,
  aliases = source.aliases,
  active_flag = source.active_flag,
  taxonomy_version = source.taxonomy_version,
  updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT *
""")

print("✓ Skill catalog written successfully (idempotent MERGE)")

# Verify the write
verify_df = spark.table(CONFIG["skill_catalog_table"])
final_count = verify_df.count()
print(f"Verified: {final_count} skills in catalog")

# Calculate metrics
new_skills_added = truly_new_df.count() if catalog_count > 0 else new_skills_df.count()
skills_with_aliases = verify_df.filter(F.size(F.col('aliases')) > 0).count()
external_sources_loaded = len([s for s in CONFIG["external_sources"].values() if s["enabled"]])

# Record sync metadata
print(f"\nRecording sync metadata to {metadata_table}...")

metadata_record = spark.createDataFrame(
    [(
        sync_version,
        run_id,
        catalog_count,
        final_count,
        new_skills_added,
        skills_with_aliases,
        external_sources_loaded,
        run_timestamp_py,
        "completed"
    )],
    metadata_schema
)

metadata_record.write.format("delta").mode("append").saveAsTable(metadata_table)
print("✓ Sync metadata recorded successfully")

# Build summary
import json

summary = {
    "status": "completed",
    "sync_version": sync_version,
    "run_id": run_id,
    "skills_before": catalog_count,
    "skills_after": final_count,
    "new_skills_added": new_skills_added,
    "skills_with_aliases": skills_with_aliases,
    "external_sources_loaded": external_sources_loaded,
    "skill_catalog_table": CONFIG["skill_catalog_table"],
    "metadata_table": metadata_table
}

print("\n" + "="*60)
print("SKILL CATALOG SYNC - SUMMARY")
print("="*60)
print(json.dumps(summary, indent=2))
print("="*60)

dbutils.notebook.exit(json.dumps(summary))

In [0]:
# Create job-skill evidence table from silver_skill_mapping
# This bridges Silver extraction to Warehouse consumption

print("\nCreating job-skill evidence table from silver skill mappings...")

try:
    # Load skill mappings from silver
    silver_mapping_df = spark.table("workspace.silver.silver_skill_mapping")
    mapping_count = silver_mapping_df.count()
    print(f"Loaded {mapping_count} skill mappings from silver")
    
    if mapping_count == 0:
        print("Warning: No skill mappings found in silver layer")
        print("Run silver_skill_extract first to populate silver_skill_mapping")
    else:
        # Join with skill catalog to get canonical skill_id
        evidence_df = silver_mapping_df.alias("m").join(
            updated_catalog_df.alias("c"),
            F.lower(F.trim(F.col("m.skill_name_normalized"))) == F.lower(F.trim(F.col("c.skill_name"))),
            "left"
        ).select(
            F.expr("uuid()").alias("evidence_id"),
            F.col("m.enterprise_job_id"),
            F.coalesce(F.col("c.canonical_skill_id"), 
                      F.concat(F.lit("skill_"), F.md5(F.col("m.skill_name_normalized")))).alias("skill_id"),
            F.col("m.skill_name_normalized").alias("skill_name"),
            F.col("m.confidence").cast("decimal(5,4)").alias("confidence_score"),
            F.col("m.extraction_method").alias("evidence_type"),
            F.when(F.col("m.extraction_method") == "KEYWORD", 1)
             .when(F.col("m.extraction_method") == "PATTERN", 2)
             .when(F.col("m.extraction_method") == "NLP", 3)
             .otherwise(4).alias("source_priority"),
            F.col("m.evidence_text"),
            F.col("m.extracted_at")
        )
        
        # Write to semantic evidence table
        evidence_table = CONFIG["skill_evidence_table"]
        print(f"Writing {evidence_df.count()} evidence records to {evidence_table}...")
        
        evidence_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(evidence_table)
        
        print("✓ Job-skill evidence table created successfully")
        
        # Show sample
        print("\nSample evidence records:")
        display(spark.table(evidence_table).limit(10))
        
except Exception as e:
    print(f"Error creating evidence table: {e}")
    print("This is expected if silver_skill_mapping doesn't exist yet")